# Data Mining Assignment - Apache Spark Data Lake Preprocessing
**Dataset:** `sales.csv` | **Environment:** Google Colab + PySpark

Name : Ajay

Roll Number : LCS2023025

In [ ]:
!pip install -q pyspark

## 1. Create the Data Lake structure
Upload `sales.csv` when prompted (skipped if it is already in `/content`). The file is **copied byte-for-byte** into Bronze and never modified.

In [2]:
import os, shutil, glob

BASE   = "data_lake"
BRONZE = f"{BASE}/bronze"
SILVER = f"{BASE}/silver"
SILVER_DIR = f"{SILVER}/preprocessed_sales"
for d in (BRONZE, SILVER_DIR):
    os.makedirs(d, exist_ok=True)

# Get sales.csv (upload in Colab if it isn't already present)
if not os.path.exists("sales.csv") and not os.path.exists(f"{BRONZE}/sales.csv"):
    from google.colab import files
    files.upload()          # choose sales.csv

if os.path.exists("sales.csv"):
    shutil.copyfile("sales.csv", f"{BRONZE}/sales.csv")   # raw copy, unchanged

for root, dirs, fs in os.walk(BASE):
    print(root, fs)

Saving sales.csv to sales.csv
data_lake []
data_lake/bronze ['sales.csv']
data_lake/silver []
data_lake/silver/preprocessed_sales []


## 2. Load and inspect with Spark
Everything is read as **string** first so that bad values (e.g. `not-a-date`) are not silently lost by schema inference.

In [3]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window

spark = (SparkSession.builder.appName("SalesDataLake")
         .master("local[*]")
         .config("spark.sql.ansi.enabled", "false")
         .config("spark.sql.legacy.timeParserPolicy", "CORRECTED")
         .getOrCreate())

raw = spark.read.option("header", True).option("inferSchema", False).csv(f"{BRONZE}/sales.csv")
raw.printSchema()
raw.show(5, truncate=False)
rows_before = raw.count()
print("Total records (Bronze):", rows_before)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- discount_percent: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)

+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name |product      |category|quantity|unit_price|discount_percent|payment_method|city      |state         |order_date|order_status|
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+---

### 2.1 Duplicates

In [4]:
exact_dups = rows_before - raw.dropDuplicates().count()
print("Exact duplicate rows (extra copies):", exact_dups)

print("order_ids appearing more than once:")
raw.groupBy("order_id").count().filter("count > 1").orderBy(F.desc("count")).show(10)

Exact duplicate rows (extra copies): 10
order_ids appearing more than once:
+--------+-----+
|order_id|count|
+--------+-----+
|  100297|    2|
|  100474|    2|
|  100692|    2|
|  100305|    2|
|  100072|    2|
|  100588|    2|
|  100434|    2|
|  100444|    2|
|  100064|    2|
|  100145|    2|
+--------+-----+
only showing top 10 rows


### 2.2 Null / empty values

In [5]:
raw.select([F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), 1).otherwise(0)).alias(c)
            for c in raw.columns]).show(vertical=True)

-RECORD 0---------------
 order_id         | 0   
 customer_id      | 0   
 customer_name    | 10  
 product          | 0   
 category         | 0   
 quantity         | 0   
 unit_price       | 0   
 discount_percent | 0   
 payment_method   | 8   
 city             | 8   
 state            | 0   
 order_date       | 0   
 order_status     | 0   



### 2.3 Text-format inconsistencies (spaces / capitalization)

In [6]:
text_cols = ["customer_name", "product", "category", "payment_method", "city", "state", "order_status"]

# leading/trailing spaces and case variants (values whose normalized form maps to >1 raw spelling)
for c in text_cols:
    spaces = raw.filter(F.col(c) != F.trim(F.col(c))).count()
    variants = (raw.filter(F.col(c).isNotNull())
                   .groupBy(F.lower(F.trim(F.col(c))).alias("norm"))
                   .agg(F.countDistinct(c).alias("spellings")).filter("spellings > 1").count())
    print(f"{c:15s} values with extra spaces: {spaces:3d} | values with multiple spellings: {variants}")

print("\nCategory spellings:")
raw.groupBy("category").count().orderBy("category").show(20, truncate=False)
print("City spellings:")
raw.groupBy("city").count().orderBy("city").show(30, truncate=False)

customer_name   values with extra spaces:   3 | values with multiple spellings: 3
product         values with extra spaces:   0 | values with multiple spellings: 0
category        values with extra spaces:   0 | values with multiple spellings: 4
payment_method  values with extra spaces:   0 | values with multiple spellings: 0
city            values with extra spaces:   3 | values with multiple spellings: 5
state           values with extra spaces:   0 | values with multiple spellings: 0
order_status    values with extra spaces:   0 | values with multiple spellings: 0

Category spellings:
+---------------+-----+
|category       |count|
+---------------+-----+
|Beauty         |136  |
|Books          |128  |
|CLOTHING       |1    |
|Clothing       |108  |
|ELECTRONICS    |1    |
|Electronics    |103  |
|Furniture      |132  |
|Grocery        |136  |
|Home Appliances|115  |
|SPORTS         |1    |
|Sports         |136  |
|beauty         |2    |
|sports         |1    |
+---------------+----

### 2.4 Invalid numerical values

In [7]:
num = raw.select("order_id",
                F.col("quantity").cast("int").alias("quantity"),
                F.col("unit_price").cast("double").alias("unit_price"),
                F.col("discount_percent").cast("double").alias("discount_percent"))

print("Non-numeric (cannot cast) values:")
num_raw = raw.select("quantity","unit_price","discount_percent")
for c in num_raw.columns:
    bad = raw.filter(F.col(c).isNotNull() & F.col(c).cast("double").isNull()).count()
    print(f"  {c}: {bad}")

print("\nInvalid ranges:")
num.select(
    F.sum((F.col("quantity") <= 0).cast("int")).alias("quantity<=0"),
    F.sum((F.col("unit_price") <= 0).cast("int")).alias("unit_price<=0"),
    F.sum(((F.col("discount_percent") < 0) | (F.col("discount_percent") > 100)).cast("int")).alias("discount_outside_0_100"),
).show()
num.describe().show()

Non-numeric (cannot cast) values:
  quantity: 0
  unit_price: 0
  discount_percent: 0

Invalid ranges:
+-----------+-------------+----------------------+
|quantity<=0|unit_price<=0|discount_outside_0_100|
+-----------+-------------+----------------------+
|         12|            8|                     8|
+-----------+-------------+----------------------+

+-------+------------------+------------------+------------------+------------------+
|summary|          order_id|          quantity|        unit_price|  discount_percent|
+-------+------------------+------------------+------------------+------------------+
|  count|              1000|              1000|              1000|              1000|
|   mean|        100497.072|             5.355|            5702.4|            19.948|
| stddev|287.50240424813074|2.9916425128890194|10990.194484849431|14.236782711763993|
|    min|            100001|                -2|            -500.0|              -5.0|
|    max|            101000|           

### 2.5 Inconsistent / invalid dates

In [8]:
DATE_FORMATS = ["yyyy-MM-dd", "yyyy/MM/dd", "dd/MM/yyyy", "MM-dd-yyyy"]

def parse_date(col):
    # only try each format on strings that match its exact shape, so formats cannot be confused
    return F.coalesce(
        F.when(col.rlike(r"^\d{4}-\d{2}-\d{2}$"), F.to_date(col, "yyyy-MM-dd")),
        F.when(col.rlike(r"^\d{4}/\d{2}/\d{2}$"), F.to_date(col, "yyyy/MM/dd")),
        F.when(col.rlike(r"^\d{2}/\d{2}/\d{4}$"), F.to_date(col, "dd/MM/yyyy")),
        F.when(col.rlike(r"^\d{2}-\d{2}-\d{4}$"), F.to_date(col, "MM-dd-yyyy")),
    )

dates = raw.select("order_id", F.trim("order_date").alias("order_date")) \
           .withColumn("parsed", parse_date(F.col("order_date")))

standard = F.col("order_date").rlike(r"^\d{4}-\d{2}-\d{2}$")
print("Non-standard (not yyyy-MM-dd) formats:", dates.filter(~standard).count())
print("Dates that cannot be parsed to a real date:", dates.filter(F.col("parsed").isNull()).count())
dates.filter(~standard).select("order_date", "parsed").groupBy("order_date", "parsed").count().show(truncate=False)

Non-standard (not yyyy-MM-dd) formats: 4
Dates that cannot be parsed to a real date: 4
+----------+----------+-----+
|order_date|parsed    |count|
+----------+----------+-----+
|31/02/2026|NULL      |1    |
|not-a-date|NULL      |1    |
|2025/07/14|2025-07-14|1    |
|09-15-2026|2026-09-15|1    |
+----------+----------+-----+



## 3. Preprocessing -> Silver layer

| Issue | Treatment |
|---|---|
| Exact duplicates | `dropDuplicates()` |
| Missing `customer_name`, `payment_method`, `city` | filled with `"Unknown"` (records preserved) |
| Extra spaces / inconsistent case | `trim`, `initcap` on name, category, city, payment method, state, status |
| Invalid `quantity` (<= 0) / `unit_price` (<= 0) | set to null, then imputed with the **median of the product's category** (fallback: overall median); flagged |
| Invalid `discount_percent` (<0 or >100) | set to **0** (no discount); flagged |
| Mixed date formats | converted to `yyyy-MM-dd` |
| Impossible dates (e.g. 31/02/2026, `not-a-date`) | left as **null** (a date should not be invented); record kept and flagged |

Records are preserved wherever possible; flag columns record every fix.

In [9]:
df = raw.dropDuplicates()

# --- text cleaning -----------------------------------------------------------
for c in ["customer_name", "category", "payment_method", "city", "state", "order_status"]:
    df = df.withColumn(c, F.initcap(F.trim(F.regexp_replace(F.col(c), r"\s+", " "))))
df = df.withColumn("product", F.trim(F.regexp_replace(F.col("product"), r"\s+", " ")))
df = df.withColumn("customer_id", F.upper(F.trim("customer_id")))

df = df.withColumn("payment_method", F.regexp_replace(F.regexp_replace("payment_method", "Cash On Delivery", "Cash on Delivery"), "^Upi$", "UPI"))

# --- missing text values ------------------------------------------------------
for c in ["customer_name", "payment_method", "city"]:
    df = df.withColumn(c, F.when(F.col(c).isNull() | (F.col(c) == ""), F.lit("Unknown")).otherwise(F.col(c)))

# --- numeric validation -------------------------------------------------------
df = (df.withColumn("quantity", F.col("quantity").cast("int"))
        .withColumn("unit_price", F.col("unit_price").cast("double"))
        .withColumn("discount_percent", F.col("discount_percent").cast("double")))

bad_q = F.col("quantity").isNull() | (F.col("quantity") <= 0)
bad_p = F.col("unit_price").isNull() | (F.col("unit_price") <= 0)
bad_d = F.col("discount_percent").isNull() | (F.col("discount_percent") < 0) | (F.col("discount_percent") > 100)

df = (df.withColumn("quantity_fixed", bad_q)
        .withColumn("unit_price_fixed", bad_p)
        .withColumn("discount_fixed", bad_d)
        .withColumn("quantity",   F.when(bad_q, None).otherwise(F.col("quantity")))
        .withColumn("unit_price", F.when(bad_p, None).otherwise(F.col("unit_price")))
        .withColumn("discount_percent", F.when(bad_d, F.lit(0.0)).otherwise(F.col("discount_percent"))))

# medians (of valid values) per category, with overall median as fallback
med = (df.groupBy("category").agg(F.percentile_approx("quantity", 0.5).alias("med_q"),
                                  F.percentile_approx("unit_price", 0.5).alias("med_p")))
overall = df.agg(F.percentile_approx("quantity", 0.5).alias("q"),
                 F.percentile_approx("unit_price", 0.5).alias("p")).first()

df = (df.join(med, "category", "left")
        .withColumn("quantity",   F.coalesce("quantity",   "med_q", F.lit(overall["q"])).cast("int"))
        .withColumn("unit_price", F.coalesce("unit_price", "med_p", F.lit(overall["p"])))
        .drop("med_q", "med_p"))

# --- dates ----------------------------------------------------------------------
d = F.trim(F.col("order_date"))
df = (df.withColumn("order_date_parsed", parse_date(d))
        .withColumn("order_date_invalid", F.col("order_date_parsed").isNull())
        .withColumn("order_date", F.date_format("order_date_parsed", "yyyy-MM-dd"))
        .drop("order_date_parsed"))

silver = df.select("order_id","customer_id","customer_name","product","category","quantity","unit_price",
                   "discount_percent","payment_method","city","state","order_date","order_status",
                   "quantity_fixed","unit_price_fixed","discount_fixed","order_date_invalid")
silver.cache()
rows_after = silver.count()
silver.printSchema()
silver.show(5, truncate=False)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = false)
 |-- unit_price: double (nullable = false)
 |-- discount_percent: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- quantity_fixed: boolean (nullable = true)
 |-- unit_price_fixed: boolean (nullable = true)
 |-- discount_fixed: boolean (nullable = true)
 |-- order_date_invalid: boolean (nullable = false)

+--------+-----------+--------------+----------------------+-----------+--------+----------+----------------+----------------+---------+-------------+----------+------------+--------------+----------------+--------------+------------------+
|order_id|cus

## 3.1 Save to Silver
Spark output folder (Parquet) inside `silver/preprocessed_sales/`, plus a single-file CSV `silver/sales_silver.csv` (required for the GitHub submission).

In [10]:
# Spark output (can contain several part files)
silver.write.mode("overwrite").parquet(SILVER_DIR)

# Single CSV for submission
tmp = f"{SILVER}/_tmp_csv"
silver.coalesce(1).write.mode("overwrite").option("header", True).csv(tmp)
part = glob.glob(f"{tmp}/part-*.csv")[0]
shutil.move(part, f"{SILVER}/sales_silver.csv")
shutil.rmtree(tmp)

for root, dirs, fs in os.walk(SILVER):
    print(root, fs)

data_lake/silver ['sales_silver.csv']
data_lake/silver/preprocessed_sales ['.part-00134-d580278a-d29d-4791-b0bf-2c1cd3c000d6-c000.snappy.parquet.crc', '.part-00160-d580278a-d29d-4791-b0bf-2c1cd3c000d6-c000.snappy.parquet.crc', '.part-00080-d580278a-d29d-4791-b0bf-2c1cd3c000d6-c000.snappy.parquet.crc', 'part-00171-d580278a-d29d-4791-b0bf-2c1cd3c000d6-c000.snappy.parquet', '.part-00186-d580278a-d29d-4791-b0bf-2c1cd3c000d6-c000.snappy.parquet.crc', 'part-00160-d580278a-d29d-4791-b0bf-2c1cd3c000d6-c000.snappy.parquet', 'part-00189-d580278a-d29d-4791-b0bf-2c1cd3c000d6-c000.snappy.parquet', '.part-00038-d580278a-d29d-4791-b0bf-2c1cd3c000d6-c000.snappy.parquet.crc', '.part-00131-d580278a-d29d-4791-b0bf-2c1cd3c000d6-c000.snappy.parquet.crc', '.part-00091-d580278a-d29d-4791-b0bf-2c1cd3c000d6-c000.snappy.parquet.crc', '.part-00037-d580278a-d29d-4791-b0bf-2c1cd3c000d6-c000.snappy.parquet.crc', 'part-00092-d580278a-d29d-4791-b0bf-2c1cd3c000d6-c000.snappy.parquet', 'part-00084-d580278a-d29d-4791-b0

## 4. Verification

In [11]:
print("Rows BEFORE preprocessing (Bronze):", rows_before)
print("Rows AFTER  preprocessing (Silver):", rows_after)
print("Rows removed (exact duplicates)   :", rows_before - rows_after)

# Read Silver back with Spark
check = spark.read.parquet(SILVER_DIR)
print("Silver rows read back:", check.count())
check.show(10, truncate=False)

csv_check = spark.read.option("header", True).csv(f"{SILVER}/sales_silver.csv")
print("CSV rows read back:", csv_check.count())

print("\nRemaining nulls in Silver (should only be order_date for impossible dates):")
check.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in check.columns[:13]]).show(vertical=True)
print("Fixes applied (counts from flag columns):")
check.select(F.sum(F.col("quantity_fixed").cast("int")).alias("quantity_imputed"),
             F.sum(F.col("unit_price_fixed").cast("int")).alias("unit_price_imputed"),
             F.sum(F.col("discount_fixed").cast("int")).alias("discount_reset"),
             F.sum(F.col("order_date_invalid").cast("int")).alias("invalid_dates_nulled")).show()

Rows BEFORE preprocessing (Bronze): 1000
Rows AFTER  preprocessing (Silver): 990
Rows removed (exact duplicates)   : 10
Silver rows read back: 990
+--------+-----------+----------------+-----------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+--------------+----------------+--------------+------------------+
|order_id|customer_id|customer_name   |product          |category       |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|quantity_fixed|unit_price_fixed|discount_fixed|order_date_invalid|
+--------+-----------+----------------+-----------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+--------------+----------------+--------------+------------------+
|100244  |C0286      |Rahul Naidu     |Football         |Sports         |2       |2320.0    |12.0            |Credit Card

## 5. Summary of preprocessing performed
1. **Bronze** keeps the raw `sales.csv` unchanged.
2. Removed exact duplicate rows.
3. Filled missing customer name, payment method and city with `Unknown`.
4. Trimmed spaces, collapsed repeated whitespace and standardized capitalization (`initcap`) for text fields.
5. Validated quantity, unit price (must be > 0) and discount (0-100): invalid quantity/price were imputed with category medians, invalid discounts were set to 0; all fixes are flagged.
6. Converted all valid dates to `yyyy-MM-dd`; impossible dates were set to null (record kept, flagged).
7. Saved the result to **Silver** as Parquet (`silver/preprocessed_sales/`) and CSV (`silver/sales_silver.csv`).